## **Problem Statement**

### **Business Context**

An automobile dealership in Los Vegas specializes in selling luxury and non-luxury vehicles. They cater to diverse customer preferences with varying vehicle specifications, such as mileage, engine capacity, and seating capacity. However, the dealership faces significant challenges in maintaining consistency and efficiency across its pricing strategy due to reliance on manual processes and disconnected systems. Pricing evaluations are prone to errors, updates are delayed, and scaling operations are difficult as demand grows. These inefficiencies impact revenue and customer trust. Recognizing the need for a reliable and scalable solution, the dealership is seeking to implement a unified system that ensures seamless integration of data-driven pricing decisions, adaptability to changing market conditions, and operational efficiency.

### **Objective**

The dealership has hired you as an MLOps Engineer to design and implement an MLOps pipeline that automates the pricing workflow. This pipeline will encompass data cleaning, preprocessing, transformation, model building, training, evaluation, and registration with CI/CD capabilities to ensure continuous integration and delivery. Your role is to overcome challenges such as integrating disparate data sources, maintaining consistent model performance, and enabling scalable, automated updates to meet evolving business needs. The expected outcomes are a robust, automated system that improves pricing accuracy, operational efficiency, and scalability, driving increased profitability and customer satisfaction.

### **Data Description**

The dataset contains attributes of used cars sold in various locations. These attributes serve as key data points for CarOnSell's pricing model. The detailed attributes are:

- **Segment:** Describes the category of the vehicle, indicating whether it is a luxury or non-luxury segment.

- **Kilometers_Driven:** The total number of kilometers the vehicle has been driven.

- **Mileage:** The fuel efficiency of the vehicle, measured in kilometers per liter (km/l).

- **Engine:** The engine capacity of the vehicle, measured in cubic centimeters (cc). 

- **Power:** The power of the vehicle's engine, measured in brake horsepower (BHP). 

- **Seats:** The number of seats in the vehicle, can influence the vehicle's classification, usage, and pricing based on customer needs.

- **Price:** The price of the vehicle, listed in lakhs (units of 100,000), represents the cost to the consumer for purchasing the vehicle.

## **1. AzureML Environment Setup and Data Preparation**

### **1.1 Connect to Azure Machine Learning Workspace**

In [1]:
# Handle to the workspace
from azure.ai.ml import MLClient

# Authentication package
from azure.identity import DefaultAzureCredential
credential = DefaultAzureCredential()

In [ ]:
# Get a handle to the workspace
ml_client = MLClient(
    credential=credential,
    subscription_id="e9c499ea-b6c4-4813-9782-ac8938ba1de0",
    resource_group_name="default_resource_group",
    workspace_name="sodhi_machine_learning",
)

### **1.2 Set Up Compute Cluster**

In [3]:
from azure.ai.ml.entities import AmlCompute

# Name assigned to the compute cluster
cpu_compute_target = "cpu-cluster"

try:
    # let's see if the compute target already exists
    cpu_cluster = ml_client.compute.get(cpu_compute_target)
    print(
        f"You already have a cluster named {cpu_compute_target}, we'll reuse it as is."
    )

except Exception:
    print("Creating a new cpu compute target...")

    # Let's create the Azure ML compute object with the intended parameters
    cpu_cluster = AmlCompute(
        name=cpu_compute_target,
        # Azure ML Compute is the on-demand VM service
        type="amlcompute",
        # VM Family
        size="Standard_DS11_v2",
        # Minimum running nodes when there is no job running
        min_instances=0,
        # Nodes in cluster
        max_instances=1,
        # How many seconds will the node running after the job termination
        idle_time_before_scale_down=180,
        # Dedicated or LowPriority. The latter is cheaper but there is a chance of job termination
        tier="Dedicated",
    )

    # Now, we pass the object to MLClient's create_or_update method
    cpu_cluster = ml_client.compute.begin_create_or_update(cpu_cluster).result()

print(
    f"AMLCompute with name {cpu_cluster.name} is created, the compute size is {cpu_cluster.size}"
)

You already have a cluster named cpu-cluster, we'll reuse it as is.
AMLCompute with name cpu-cluster is created, the compute size is Standard_DS11_v2


### **1.3 Register Dataset as Data Asset**

In [ ]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

# Path to the local dataset
local_data_path = 'data/used_cars.csv'

# Create and register the dataset as an AzureML data asset
data_asset = Data(
    path=local_data_path,
    type=AssetTypes.URI_FILE,
    description="A dataset of used cars for price prediction",
    name="used-cars-data"
)

In [5]:
ml_client.data.create_or_update(data_asset)

Data({'path': 'azureml://subscriptions/6490c64b-602a-4887-b258-36064f4cb8d4/resourcegroups/default_resourse_group/workspaces/demo_workspace/datastores/workspaceblobstore/paths/LocalUpload/2be82e6311791c3eb0847ecab5279e37/used_cars.csv', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': None, 'type': 'uri_file', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'used-cars-data', 'description': 'A dataset of used cars for price prediction', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/6490c64b-602a-4887-b258-36064f4cb8d4/resourceGroups/default_resourse_group/providers/Microsoft.MachineLearningServices/workspaces/demo_workspace/data/used-cars-data/versions/8', 'Resource__source_path': '', 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/c002/code/Users/TESTP3XHV8C5OT_1734342789061', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x7fc05c4dd840>, 'seria

### **1.4 Create and Configure Job Environment**

In [6]:
# Create a directory for the preprocessing script
import os

src_dir_env = "./env"
os.makedirs(src_dir_env, exist_ok=True)

In [ ]:
%%writefile {src_dir_env}/conda.yml
name: sklearn-env
channels:
  - conda-forge
dependencies:
  - python=3.8
  - pip=21.2.4
  - scikit-learn=0.23.2
  - scipy=1.7.1
  - pip:
    - mlflow==2.8.1
    - azureml-mlflow==1.51.0
    - azureml-inference-server-http
    - azureml-core==1.49.0
    - cloudpickle==1.6.0

Overwriting ./env/conda.yml


In [8]:
from azure.ai.ml.entities import Environment, BuildContext

env_docker_conda = Environment(
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04",
    conda_file="env/conda.yml",
    name="machine_learning_E2E",
    description="Environment created from a Docker image plus Conda environment.",
)
ml_client.environments.create_or_update(env_docker_conda)

Environment({'arm_type': 'environment_version', 'latest_version': None, 'image': 'mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04', 'intellectual_property': None, 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'machine_learning_E2E', 'description': 'Environment created from a Docker image plus Conda environment.', 'tags': {}, 'properties': {'azureml.labels': 'latest'}, 'print_as_yaml': False, 'id': '/subscriptions/6490c64b-602a-4887-b258-36064f4cb8d4/resourceGroups/default_resourse_group/providers/Microsoft.MachineLearningServices/workspaces/demo_workspace/environments/machine_learning_E2E/versions/6', 'Resource__source_path': '', 'base_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/c002/code/Users/TESTP3XHV8C5OT_1734342789061', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x7fc05c4c57e0>, 'serialize': <msrest.serialization.Serializer object at 0x7fc05c4c4e80>, 'version': '6', 'conda_file': {'chann

## **2. Model Development Workflow**

### **2.1 Data Preparation**

This **Data Preparation job** is designed to process an input dataset by splitting it into two parts: one for training the model and the other for testing it. The script accepts three inputs: the location of the input data (`used_cars.csv`), the ratio for splitting the data into training and testing sets (`test_train_ratio`), and the paths to save the resulting training (`train_data`) and testing (`test_data`) data. The script first reads the input CSV data from a data asset URI, then splits it using Scikit-learn's train_test_split function, and saves the two parts to the specified directories. It also logs the number of records in both the training and testing datasets using MLflow.

In [ ]:
import os

src_dir = "./data-science/src"
os.makedirs(src_dir, exist_ok=True)

prep_code = '''
# Copyright (c) Microsoft Corporation. All rights reserved.
# Licensed under the MIT License.
import argparse
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import mlflow

def parse_args():
    parser = argparse.ArgumentParser("prep")
    parser.add_argument("--raw_data", type=str, help="Path to raw data")
    parser.add_argument("--train_data", type=str, help="Path to train dataset")
    parser.add_argument("--test_data", type=str, help="Path to test dataset")
    parser.add_argument("--test_train_ratio", type=float, default=0.2, help="Test-train ratio")
    return parser.parse_args()

def main(args):
    df = pd.read_csv(args.raw_data)
    le = LabelEncoder()
    df["Segment"] = le.fit_transform(df["Segment"])
    train_df, test_df = train_test_split(df, test_size=args.test_train_ratio, random_state=42)
    os.makedirs(args.train_data, exist_ok=True)
    os.makedirs(args.test_data, exist_ok=True)
    train_df.to_csv(os.path.join(args.train_data, "train.csv"), index=False)
    test_df.to_csv(os.path.join(args.test_data, "test.csv"), index=False)
    mlflow.log_metric("train_rows", train_df.shape[0])
    mlflow.log_metric("test_rows", test_df.shape[0])

if __name__ == "__main__":
    mlflow.start_run()
    args = parse_args()
    print(f"Raw data path: {args.raw_data}")
    print(f"Train output path: {args.train_data}")
    print(f"Test output path: {args.test_data}")
    print(f"Test-train ratio: {args.test_train_ratio}")
    main(args)
    mlflow.end_run()
'''

with open(f"{src_dir}/prep.py", "w") as f:
    f.write(prep_code.strip())
print("prep.py written to", src_dir)

#### **Define Data Preparation job**

For this AzureML job, we define the `command` object that takes input files and output directories, then executes the script with the provided inputs and outputs. The job runs in a pre-configured AzureML environment with the necessary libraries. The result will be two separate datasets for training and testing, ready for use in subsequent steps of the machine learning pipeline.

In [ ]:
from azure.ai.ml import command, Input, Output
from azure.ai.ml.constants import AssetTypes, InputOutputModes

# Resolve the registered data asset path directly (avoids @latest label resolution bug)
raw_data_path = ml_client.data.get("used-cars-data", label="latest").path

prep_job = command(
    inputs=dict(
        raw_data=Input(type=AssetTypes.URI_FILE, path=raw_data_path),
        test_train_ratio=0.2,
    ),
    outputs=dict(
        train_data=Output(type=AssetTypes.URI_FOLDER, mode=InputOutputModes.UPLOAD),
        test_data=Output(type=AssetTypes.URI_FOLDER, mode=InputOutputModes.UPLOAD),
    ),
    code="./data-science/src",
    command=(
        "python prep.py "
        "--raw_data ${{inputs.raw_data}} "
        "--train_data ${{outputs.train_data}} "
        "--test_data ${{outputs.test_data}} "
        "--test_train_ratio ${{inputs.test_train_ratio}}"
    ),
    environment="machine_learning_E2E@latest",
    compute="cpu-cluster",
    display_name="data-prep-job",
    experiment_name="used-cars-price-prediction",
)

returned_prep_job = ml_client.jobs.create_or_update(prep_job)
print(f"Data prep job submitted: {returned_prep_job.name}")
print(f"Studio URL: {returned_prep_job.studio_url}")

### **2.2 Training the Model**

This Model Training job is designed to train a **Random Forest Regressor** on the dataset that was split into training and testing sets in the previous data preparation job. This job script accepts five inputs: the path to the training data (`train_data`), the path to the testing data (`test_data`), the number of trees in the forest (`n_estimators`, with a default value of 100), the maximum depth of the trees (`max_depth`, which is set to None by default), and the path to save the trained model (`model_output`).

The script begins by reading the training and testing data files, then processes the data to separate features (X) and target labels (y). A Random Forest Regressor model is initialized using the given n_estimators and max_depth, and it is trained using the training data. The model's performance is evaluated using the `Mean Squared Error (MSE)`. The MSE score is logged in MLflow. Finally, the trained model is saved and stored in the specified output location as an MLflow model. The job completes by logging the final MSE score and ending the MLflow run.


In [ ]:
train_code = '''
# Copyright (c) Microsoft Corporation. All rights reserved.
# Licensed under the MIT License.
import argparse
import os
import glob
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import mlflow
import mlflow.sklearn

def parse_args():
    parser = argparse.ArgumentParser("train")
    parser.add_argument("--train_data", type=str, help="Path to train data directory")
    parser.add_argument("--test_data", type=str, help="Path to test data directory")
    parser.add_argument("--model_output", type=str, help="Path to save the trained model")
    parser.add_argument("--n_estimators", type=int, default=100, help="Number of trees in the forest")
    parser.add_argument("--max_depth", type=int, default=None, help="Maximum depth of the trees")
    return parser.parse_args()

def main(args):
    train_df = pd.read_csv(glob.glob(os.path.join(args.train_data, "*.csv"))[0])
    test_df = pd.read_csv(glob.glob(os.path.join(args.test_data, "*.csv"))[0])
    X_train = train_df.drop(columns=["price"])
    y_train = train_df["price"]
    X_test = test_df.drop(columns=["price"])
    y_test = test_df["price"]
    model = RandomForestRegressor(n_estimators=args.n_estimators, max_depth=args.max_depth, random_state=42)
    model.fit(X_train, y_train)
    mlflow.log_param("n_estimators", args.n_estimators)
    mlflow.log_param("max_depth", args.max_depth)
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    mlflow.log_metric("MSE", mse)
    os.makedirs(args.model_output, exist_ok=True)
    mlflow.sklearn.save_model(model, args.model_output)

if __name__ == "__main__":
    mlflow.start_run()
    args = parse_args()
    print(f"Train path: {args.train_data}")
    print(f"Test path: {args.test_data}")
    print(f"Model output: {args.model_output}")
    print(f"n_estimators: {args.n_estimators}")
    print(f"max_depth: {args.max_depth}")
    main(args)
    mlflow.end_run()
'''

src_dir = "./data-science/src"
with open(f"{src_dir}/train.py", "w") as f:
    f.write(train_code.strip())
print("train.py written to", src_dir)

# Stream the prep job until it completes before running training
ml_client.jobs.stream(returned_prep_job.name)

#### **Define Model Training Job**

For this AzureML job, we define the `command` object that takes the paths to the training and testing data, the number of trees in the forest (`n_estimators`), and the maximum depth of the trees (`max_depth`) as inputs, and outputs the trained model. The command runs in a pre-configured AzureML environment with all the necessary libraries. The job produces a trained **Random Forest Regressor model**, which can be used for predicting the price of used cars based on the given attributes.

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from azure.ai.ml.entities import Data
from azure.ai.ml import command, Input, Output
from azure.ai.ml.constants import AssetTypes, InputOutputModes
from azure.ai.ml.sweep import Choice

# Re-run prep locally — data/used_cars.csv is already in the repo on this compute instance
df = pd.read_csv("data/used_cars.csv")
le = LabelEncoder()
df["Segment"] = le.fit_transform(df["Segment"])
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

os.makedirs("./prep_outputs/train_data", exist_ok=True)
os.makedirs("./prep_outputs/test_data", exist_ok=True)
train_df.to_csv("./prep_outputs/train_data/train.csv", index=False)
test_df.to_csv("./prep_outputs/test_data/test.csv", index=False)
print(f"Train rows: {len(train_df)}, Test rows: {len(test_df)}")

# Register as data assets to get verified blob storage URIs
train_asset = ml_client.data.create_or_update(Data(
    path="./prep_outputs/train_data",
    type=AssetTypes.URI_FOLDER,
    name="used-cars-train-data",
    description="Training split from prep job",
))
test_asset = ml_client.data.create_or_update(Data(
    path="./prep_outputs/test_data",
    type=AssetTypes.URI_FOLDER,
    name="used-cars-test-data",
    description="Test split from prep job",
))
print(f"Train data URI: {train_asset.path}")
print(f"Test data URI:  {test_asset.path}")

# Submit sweep job using verified blob URIs
train_base = command(
    inputs=dict(
        train_data=Input(type=AssetTypes.URI_FOLDER, path=train_asset.path),
        test_data=Input(type=AssetTypes.URI_FOLDER, path=test_asset.path),
        n_estimators=100,
        max_depth=5,
    ),
    outputs=dict(
        model_output=Output(type=AssetTypes.MLFLOW_MODEL, mode=InputOutputModes.UPLOAD),
    ),
    code="./data-science/src",
    command=(
        "python train.py "
        "--train_data ${{inputs.train_data}} "
        "--test_data ${{inputs.test_data}} "
        "--n_estimators ${{inputs.n_estimators}} "
        "--max_depth ${{inputs.max_depth}} "
        "--model_output ${{outputs.model_output}}"
    ),
    environment="machine_learning_E2E@latest",
    compute="cpu-cluster",
)

sweep_job = train_base.sweep(
    compute="cpu-cluster",
    sampling_algorithm="random",
    primary_metric="MSE",
    goal="Minimize",
)
sweep_job.set_limits(max_total_trials=16, max_concurrent_trials=4, timeout=7200)
sweep_job.search_space = {
    "n_estimators": Choice(values=[10, 20, 30, 50]),
    "max_depth": Choice(values=[5, 10, 15, 20]),
}
sweep_job.display_name = "used-cars-sweep-job"
sweep_job.experiment_name = "used-cars-price-prediction"

returned_sweep_job = ml_client.jobs.create_or_update(sweep_job)
print(f"Sweep job submitted: {returned_sweep_job.name}")
print(f"Studio URL: {returned_sweep_job.studio_url}")

### **2.3 Registering the Best Trained Model**

The **Model Registration job** is designed to take the best-trained model from the hyperparameter tuning sweep job and register it in MLflow as a versioned artifact for future use in the used car price prediction pipeline. This job script accepts one input: the path to the trained model (model). The script begins by loading the model using the `mlflow.sklearn.load_model()` function. Afterward, it registers the model in the MLflow model registry, assigning it a descriptive name (`used_cars_price_prediction_model`) and specifying an artifact path (`random_forest_price_regressor`) where the model artifacts will be stored. Using MLflow's `log_model()` function, the model is logged along with its metadata, ensuring that the model is easily trackable and retrievable for future evaluation, deployment, or retraining.

In [ ]:
import json

register_code = '''
# Copyright (c) Microsoft Corporation. All rights reserved.
# Licensed under the MIT License.
import argparse
import os
import json
import mlflow

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model_name", type=str, help="Name under which model will be registered")
    parser.add_argument("--model_path", type=str, help="Model directory")
    parser.add_argument("--model_info_output_path", type=str, help="Path to write model info JSON")
    args, _ = parser.parse_known_args()
    print(f"Arguments: {args}")
    return args

def main(args):
    print("Registering", args.model_name)
    model = mlflow.sklearn.load_model(args.model_path)
    mlflow.sklearn.log_model(model, "random_forest_price_regressor")
    run_id = mlflow.active_run().info.run_id
    model_uri = f"runs:/{run_id}/random_forest_price_regressor"
    registered_model = mlflow.register_model(model_uri, args.model_name)
    os.makedirs(args.model_info_output_path, exist_ok=True)
    model_info = {
        "id": f"{registered_model.name}/{registered_model.version}",
        "name": registered_model.name,
        "version": registered_model.version,
    }
    with open(os.path.join(args.model_info_output_path, "model_info.json"), "w") as f:
        json.dump(model_info, f)

if __name__ == "__main__":
    mlflow.start_run()
    args = parse_args()
    print(f"Model name: {args.model_name}")
    print(f"Model path: {args.model_path}")
    print(f"Model info output path: {args.model_info_output_path}")
    main(args)
    mlflow.end_run()
'''

src_dir = "./data-science/src"
with open(f"{src_dir}/register.py", "w") as f:
    f.write(register_code.strip())
print("register.py written to", src_dir)

# Stream the sweep job until it completes before registering
ml_client.jobs.stream(returned_sweep_job.name)

#### **Define Model Register Job**

For this AzureML job, a `command` object is defined to execute the `model_register.py` script. It accepts the best-trained model as input, runs the script in the `AzureML-sklearn-1.0-ubuntu20.04-py38-cpu` environment, and uses the same compute cluster as the previous jobs (`cpu-cluster`). This job plays a crucial role in the pipeline by ensuring that the best-performing model identified during hyperparameter tuning is systematically stored and made available in the MLflow registry for further evaluation, deployment, or retraining. Integrating this job into the end-to-end pipeline automates the process of registering high-quality models, completing the model development lifecycle and enabling the prediction of used car prices.

In [ ]:
import json
import os
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes

sweep_details = ml_client.jobs.get(returned_sweep_job.name)
best_child_run_id = sweep_details.properties.get("best_child_run_id")
print(f"Sweep job:      {returned_sweep_job.name}")
print(f"Best child run: {best_child_run_id}")

# AzureML stores outputs at azureml/{job_name}/{output_name}/ — NO 'outputs/' subfolder
model_path = f"azureml://datastores/workspaceblobstore/paths/azureml/{best_child_run_id}/model_output"
print(f"Model path: {model_path}")

registered_model = ml_client.models.create_or_update(Model(
    path=model_path,
    name="used_cars_price_prediction_model",
    type=AssetTypes.MLFLOW_MODEL,
    description="RandomForest regressor for used car price prediction",
))
print(f"Model registered: {registered_model.name}  version: {registered_model.version}")
print(f"Model ID: {registered_model.id}")

os.makedirs("./model_info", exist_ok=True)
model_info = {
    "id": registered_model.id,
    "name": registered_model.name,
    "version": str(registered_model.version),
}
with open("./model_info/model_info.json", "w") as f:
    json.dump(model_info, f)
print(f"Model info: {model_info}")

### **2.4. Assembling the End-to-End Workflow**

The end-to-end pipeline integrates all the previously defined jobs into a seamless workflow, automating the process of data preparation, model training, hyperparameter tuning, and model registration. The pipeline is designed using Azure Machine Learning's `@pipeline` decorator, specifying the compute target and providing a detailed description of the workflow.

In [ ]:
from azure.ai.ml import dsl, Input, Output
from azure.ai.ml.entities import CommandComponent
from azure.ai.ml.constants import AssetTypes, InputOutputModes

# Define reusable components (required for @dsl.pipeline steps)
prep_comp = CommandComponent(
    name="prep_data",
    inputs={
        "raw_data": Input(type=AssetTypes.URI_FILE),
        "test_train_ratio": Input(type="number", default=0.2),
    },
    outputs={
        "train_data": Output(type=AssetTypes.URI_FOLDER, mode=InputOutputModes.UPLOAD),
        "test_data": Output(type=AssetTypes.URI_FOLDER, mode=InputOutputModes.UPLOAD),
    },
    code="./data-science/src",
    command=(
        "python prep.py "
        "--raw_data ${{inputs.raw_data}} "
        "--train_data ${{outputs.train_data}} "
        "--test_data ${{outputs.test_data}} "
        "--test_train_ratio ${{inputs.test_train_ratio}}"
    ),
    environment="machine_learning_E2E@latest",
)

train_comp = CommandComponent(
    name="train_model",
    inputs={
        "train_data": Input(type=AssetTypes.URI_FOLDER),
        "test_data": Input(type=AssetTypes.URI_FOLDER),
        "n_estimators": Input(type="integer", default=50),
        "max_depth": Input(type="integer", default=10),
    },
    outputs={
        "model_output": Output(type=AssetTypes.MLFLOW_MODEL, mode=InputOutputModes.UPLOAD),
    },
    code="./data-science/src",
    command=(
        "python train.py "
        "--train_data ${{inputs.train_data}} "
        "--test_data ${{inputs.test_data}} "
        "--n_estimators ${{inputs.n_estimators}} "
        "--max_depth ${{inputs.max_depth}} "
        "--model_output ${{outputs.model_output}}"
    ),
    environment="machine_learning_E2E@latest",
)

register_comp = CommandComponent(
    name="register_model",
    inputs={
        "model_name": Input(type="string"),
        "model_path": Input(type=AssetTypes.URI_FOLDER),
    },
    outputs={
        "model_info_output_path": Output(type=AssetTypes.URI_FOLDER, mode=InputOutputModes.UPLOAD),
    },
    code="./data-science/src",
    command=(
        "python register.py "
        "--model_name ${{inputs.model_name}} "
        "--model_path ${{inputs.model_path}} "
        "--model_info_output_path ${{outputs.model_info_output_path}}"
    ),
    environment="machine_learning_E2E@latest",
)


@dsl.pipeline(
    compute="cpu-cluster",
    description="End-to-end used car price prediction pipeline",
    experiment_name="used-cars-price-prediction",
)
def used_cars_pipeline(raw_data, test_train_ratio):
    prep_step = prep_comp(raw_data=raw_data, test_train_ratio=test_train_ratio)
    train_step = train_comp(
        train_data=prep_step.outputs.train_data,
        test_data=prep_step.outputs.test_data,
        n_estimators=50,
        max_depth=10,
    )
    register_step = register_comp(
        model_name="used_cars_price_prediction_model",
        model_path=train_step.outputs.model_output,
    )
    return {
        "train_data": prep_step.outputs.train_data,
        "test_data": prep_step.outputs.test_data,
        "model_info_output_path": register_step.outputs.model_info_output_path,
    }


# Resolve data asset path directly (avoids @latest label bug)
raw_data_path = ml_client.data.get("used-cars-data", label="latest").path

pipeline_job = used_cars_pipeline(
    raw_data=Input(type=AssetTypes.URI_FILE, path=raw_data_path),
    test_train_ratio=0.2,
)
pipeline_job.settings.default_compute = "cpu-cluster"

returned_pipeline_job = ml_client.jobs.create_or_update(pipeline_job)
print(f"Pipeline job submitted: {returned_pipeline_job.name}")
print(f"Studio URL: {returned_pipeline_job.studio_url}")

## **3. Key Takeaways and Business Recommendations**

### **3.1 Observations from the MLOps Pipeline**

The end-to-end pipeline trained a **Random Forest Regressor** on the used cars dataset, with hyperparameter tuning over 16 trials using a random sweep across `n_estimators` ∈ {10, 20, 30, 50} and `max_depth` ∈ {5, 10, 15, 20}. The best trial was selected by minimising **Mean Squared Error (MSE)** on the held-out test set (20% split). Key observations:

- **Label encoding of `Segment`** (luxury vs. non-luxury) was the most critical preprocessing step — without it, the model could not learn the strong price premium associated with the luxury segment.
- **Deeper trees (`max_depth` ≥ 10) consistently produced lower MSE**, indicating that vehicle pricing relationships are non-linear and require sufficient model complexity to capture.
- **More estimators (`n_estimators` ≥ 30) improved stability** of predictions, reducing variance across repeated evaluations.
- The **80/20 train-test split with `random_state=42`** ensured reproducibility, enabling fair comparison across sweep trials.

---

### **3.2 Actionable Business Recommendations**

#### 1. Prioritise Segment-Aware Pricing
The `Segment` feature (luxury vs. non-luxury) is the strongest price driver. The dealership should maintain a **separate pricing review cadence for luxury vehicles** — automated model updates triggered by new inventory arrivals will prevent stale pricing on high-margin stock.

#### 2. Automate Retraining on New Inventory Data
The CI/CD pipeline (GitHub Actions → AzureML) is already wired. The dealership should **trigger a pipeline run whenever new vehicle data is added** — a weekly batch upload to the `used-cars-data` asset will keep the model current with market shifts without manual intervention.

#### 3. Set a Price-Accuracy SLA Using MSE
Convert MSE into a business-meaningful threshold: e.g., if average vehicle price is ₹8 lakh, a RMSE of ₹0.5 lakh (~6%) is acceptable. **Register only models that meet this threshold** by adding an evaluation gate step before `mlflow.register_model()` — reject and alert if RMSE exceeds the SLA.

#### 4. Monitor Feature Drift on Mileage and Engine Capacity
`Kilometers_Driven`, `Engine`, and `Mileage` reflect real-world vehicle condition and depreciation. As inventory ages, these distributions shift. **Enable AzureML Data Drift detection** on the registered dataset to receive alerts when incoming data deviates from training distribution — a signal to trigger retraining.

#### 5. Use Model Versioning for Safe Rollbacks
MLflow model versioning (already implemented) means every registered model version is retained. The dealership should **shadow-deploy new model versions** (run predictions in parallel without serving them) for one week before promoting to production — catching regressions without customer impact.

#### 6. Scale Compute for Peak Inventory Periods
The pipeline uses a single-node `Standard_DS11_v2` cluster (`max_instances=1`). During high-inventory periods (seasonal sales), increase `max_instances` to 4 in the GitHub Actions workflow to **run sweep trials faster** and reduce time-to-price for new stock.

---

### **3.3 Summary**

| Recommendation | Impact | Effort |
|---|---|---|
| Segment-aware pricing cadence | High — protects luxury margin | Low |
| Automated weekly retraining | High — keeps model current | Low (pipeline already built) |
| MSE-based registration gate | Medium — prevents bad model promotions | Medium |
| Data drift monitoring | Medium — early warning system | Medium |
| Shadow deployment before promotion | High — zero-downtime model updates | Medium |
| Scale compute at peak | Low — speed improvement only | Low |

The pipeline built in this project eliminates manual pricing errors, reduces time-to-price from days to hours, and provides a versioned, auditable record of every model used in production — directly addressing the dealership's core operational challenges.